<a href="https://colab.research.google.com/github/SlavaMarina/ib-cs-ml-course/blob/main/week2_supervised/Week2_Lesson4_Workshop.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 IB CS — Week 2 · Lesson 4 (Workshop)
## Workshop "The Loan Predictor" — Kaggle-style Competition
### A4.3.2 (Classification) + A4.3.3 (Hyperparameter tuning) + A4.2.2 (Feature selection in action)

> ⏱ Duration: ~90 minutes (a double lesson).
> 💻 Platform: Google Colab
> 🎯 Goal: work through a full **classifier training cycle** in Kaggle style: data → features → model → metrics → tuning.

---

### 📋 Workshop plan

| Part | Topic | Time |
|---|---|---|
| 1 | Load the "Loan Predictor" dataset | 10 min |
| 2 | Feature engineering & selection (A4.2.2) | 15 min |
| 3 | Train/Test split | 10 min |
| 4 | Baseline KNN model | 15 min |
| 5 | Baseline Decision Tree model | 15 min |
| 6 | Calculate all 4 metrics + Confusion Matrix | 20 min |
| 7 | 🏆 Competition: hyperparameter tuning | 25 min |
| 8 | IB-style conclusion | 10 min |

---

### ⚔️ Competition

At the end of the lesson, you will show your **best F1 score**. The winner gets **+5 bonus points** for the final homework. 😎


## Part 1 · Dataset "Loan Predictor"

> Context: a bank decides whether to **approve** or **deny** a loan. We train a model to automate this decision.


In [ ]:
# === Generate a synthetic dataset ===
# IMPORTANT: same for all students because the seed is fixed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(2027)
sns.set_style('whitegrid')

n = 1000
df = pd.DataFrame({
    'age':              np.random.normal(40, 12, n).clip(18, 75).round(0),
    'annual_income_k':  np.random.lognormal(3.8, 0.6, n).clip(15, 300).round(1),
    'employment_years': np.random.exponential(8, n).clip(0, 40).round(1),
    'credit_score':     np.random.normal(680, 80, n).clip(300, 850).round(0),
    'debt_to_income':   np.random.beta(2, 5, n).round(3),  # 0-1
    'loan_amount_k':    np.random.lognormal(3.2, 0.7, n).clip(5, 200).round(1),
    'num_credit_cards': np.random.choice([0,1,2,3,4,5,6,7], n,
                                          p=[0.05,0.2,0.3,0.2,0.1,0.08,0.05,0.02]),
    'years_education':  np.random.choice([10,12,14,16,18], n,
                                          p=[0.1,0.4,0.15,0.25,0.1]),
    'random_noise':     np.random.normal(0, 1, n),  # noise feature: should be dropped
})

# Target variable: was the loan approved?
# Logic: high credit_score + good income + low debt + moderate loan amount
score = (
    0.4 * (df['credit_score'] - 600) / 250
    + 0.25 * np.log1p(df['annual_income_k']) / np.log(300)
    - 0.3 * df['debt_to_income']
    - 0.15 * df['loan_amount_k'] / df['annual_income_k'].clip(20)
    + 0.1 * df['employment_years'] / 40
    + np.random.normal(0, 0.15, n)
)
df['loan_approved'] = (score > score.median()).astype(int)

print(f"📊 Dataset shape: {df.shape}")
print(f"   Approved (1): {df['loan_approved'].sum()} ({df['loan_approved'].mean()*100:.0f}%)")
print(f"   Denied   (0): {(1-df['loan_approved']).sum()} ({(1-df['loan_approved'].mean())*100:.0f}%)")
df.head(8)


### 🎯 TRY IT #1 — Which features seem important?

Before looking at correlations, **guess**:
- Which 3 features would you keep?
- Which 2 features would you drop?

> 💡 **Tip:** use common sense. This is a financial model; what factors genuinely affect loan approval?


## Part 2 · Feature Selection (A4.2.2) in practice

We will use a **filter method** (Pearson's r), a common IB IA approach.


In [ ]:
# Correlations with the target variable
correlations = df.corr()['loan_approved'].drop('loan_approved').abs().sort_values(ascending=False)
print("=== |Pearson's r| with loan_approved ===")
print(correlations.round(3))

# Visualisation
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['green' if v > 0.1 else 'orange' if v > 0.05 else 'red' for v in correlations]
ax.barh(correlations.index, correlations.values, color=colors, edgecolor='black')
ax.axvline(0.1, color='black', linestyle='--', label='Threshold = 0.1')
ax.set_xlabel("|Pearson's r|", fontsize=11)
ax.set_title('Filter Method: ranking features by correlation with target',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

# Full correlation heatmap
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Correlation matrix — full picture', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


### 🎯 TRY IT #2 — Decide which features to use

Looking at the correlations:

1. Which feature has the **highest** correlation with `loan_approved`?
2. Which feature is **noise** and close to 0?
3. Are there pairs of features with high correlation **with each other** (multicollinearity)?

> 💎 **IB-style justification:** *"I removed `random_noise` because its correlation with the target was below 0.05, indicating no predictive value. I retained the top X features based on a threshold of |r| > 0.1."*


In [ ]:
# Final feature set: keep features with correlation > 0.05
# In a real IA, the threshold is usually tuned with cross-validation
selected_features = correlations[correlations > 0.05].index.tolist()
print(f"✅ Keeping {len(selected_features)} features: {selected_features}")
print(f"❌ Dropping: {[c for c in correlations.index if c not in selected_features]}")


## Part 3 · Train/Test Split + Normalisation

> 💎 **Secret:** in the IB IA, you **must** have separate train and test sets. Evaluating on the same data used for training is a **methodological error**.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[selected_features]
y = df['loan_approved']

# 70% train, 30% test: a standard IB choice
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}")
print(f"Test size:  {X_test.shape}")
print(f"\n⚠️ stratify=y → preserves class proportions in train and test:")
print(f"  Train approved %: {y_train.mean()*100:.1f}%")
print(f"  Test  approved %: {y_test.mean()*100:.1f}%")

# Normalisation with StandardScaler: critical for KNN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # IMPORTANT: transform, not fit_transform on test!

print("\n✅ Normalisation complete (StandardScaler)")
print("   ⚠️ On test data, use .transform(), not .fit_transform()!")


> 🚨 **CRITICAL ERROR:** `scaler.fit_transform(X_test)` is **data leakage**. On the test set, we use the **same scaler parameters** calculated from the training set. This is a common IA mistake and can cost 1-2 marks.


## Part 4 · Baseline KNN model

> Hyperparameter: `n_neighbors` (=K). We start with K=5.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)

# Metrics
print("=== KNN (k=5) — baseline model ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_knn):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_knn):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_knn):.4f}")

print("\n=== classification_report ===")
print(classification_report(y_test, y_pred_knn, target_names=['Denied', 'Approved']))


In [ ]:
# Confusion Matrix visualisation
cm_knn = confusion_matrix(y_test, y_pred_knn)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: Denied', 'Predicted: Approved'],
            yticklabels=['Actual: Denied', 'Actual: Approved'],
            cbar=False, ax=ax, annot_kws={'size': 14})
ax.set_title('Confusion Matrix — KNN (k=5)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Identify TP, FP, TN, FN explicitly for understanding
TN, FP, FN, TP = cm_knn.ravel()
print(f"\nTN (correctly denied)    = {TN}")
print(f"FP (wrongly approved)    = {FP}  ← potentially bad loans")
print(f"FN (wrongly denied)      = {FN}  ← missed customers")
print(f"TP (correctly approved)  = {TP}")


## Part 5 · Baseline Decision Tree model

> Hyperparameter: `max_depth`. We start with depth=5.
> Advantage: it **does not require normalisation**.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# IMPORTANT: for a Decision Tree, we can use unscaled data
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)  # no scaling!

y_pred_dt = dt.predict(X_test)

print("=== Decision Tree (max_depth=5) ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_dt):.4f}")


In [ ]:
# Confusion matrix for Decision Tree
cm_dt = confusion_matrix(y_test, y_pred_dt)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Denied', 'Approved'],
            yticklabels=['Denied', 'Approved'],
            cbar=False, ax=axes[0], annot_kws={'size': 14})
axes[0].set_title('KNN (k=5)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Denied', 'Approved'],
            yticklabels=['Denied', 'Approved'],
            cbar=False, ax=axes[1], annot_kws={'size': 14})
axes[1].set_title('Decision Tree (depth=5)', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

plt.tight_layout(); plt.show()


In [ ]:
# Visualise the tree itself: a major advantage for interpretability
fig, ax = plt.subplots(figsize=(18, 9))
plot_tree(dt, feature_names=selected_features, class_names=['Denied', 'Approved'],
          filled=True, rounded=True, fontsize=8, ax=ax)
ax.set_title('Decision Tree — interpretable model (shows decision paths)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Which features did the tree use?
importances = pd.Series(dt.feature_importances_, index=selected_features).sort_values(ascending=False)
print("\n=== Feature importances (from the tree) ===")
print(importances.round(3))


> 💎 **IA secret:** `feature_importances_` from a Decision Tree is an **embedded method** for feature selection. This connects back to A4.2.2 and can be used for later feature filtering.


## Part 6 · Metric comparison — model comparison

> 📘 This previews **A4.3.10 — Model Selection**. In IB Section B, you may be asked: *"Justify which model should be deployed."*


In [ ]:
# Summary table
results = pd.DataFrame({
    'KNN (k=5)': [
        accuracy_score(y_test, y_pred_knn),
        precision_score(y_test, y_pred_knn),
        recall_score(y_test, y_pred_knn),
        f1_score(y_test, y_pred_knn),
    ],
    'Decision Tree (depth=5)': [
        accuracy_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_dt),
    ],
}, index=['Accuracy', 'Precision', 'Recall', 'F1 Score']).round(4)

print(results)

# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 5))
results.plot(kind='bar', ax=ax, color=['steelblue', 'green'], edgecolor='black')
ax.set_ylabel('Score'); ax.set_ylim(0, 1.05)
ax.set_title('KNN vs Decision Tree — metric comparison', fontsize=12, fontweight='bold')
ax.legend(loc='lower right'); ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()


## 🏆 Part 7 · COMPETITION — Hyperparameter Tuning

> **Goal:** find the **best** hyperparameters for KNN and Decision Tree.
> **Ranking metric:** F1 Score.

### 7.1 — KNN tuning


In [ ]:
# Try different K values
k_values = list(range(1, 50, 2))  # odd K from 1 to 49 to avoid ties
f1_scores_knn = []
acc_scores_knn = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    f1_scores_knn.append(f1_score(y_test, pred))
    acc_scores_knn.append(accuracy_score(y_test, pred))

best_k = k_values[np.argmax(f1_scores_knn)]
print(f"🏆 Best K = {best_k}")
print(f"   F1 = {max(f1_scores_knn):.4f}")
print(f"   Accuracy = {acc_scores_knn[np.argmax(f1_scores_knn)]:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(k_values, f1_scores_knn, 'o-', label='F1 Score', color='steelblue', linewidth=2)
ax.plot(k_values, acc_scores_knn, 's--', label='Accuracy', color='orange', alpha=0.7)
ax.axvline(best_k, color='red', linestyle=':', label=f'Best K = {best_k}')
ax.set_xlabel('K (number of neighbours)', fontsize=11)
ax.set_ylabel('Score'); ax.set_title('KNN Hyperparameter Tuning', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


### 7.2 — Decision Tree tuning

In [ ]:
# Try different max_depth values
depths = list(range(1, 21))
f1_scores_dt = []
train_acc_dt = []  # for demonstrating overfitting
test_acc_dt = []

for d in depths:
    model = DecisionTreeClassifier(max_depth=d, random_state=42)
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    f1_scores_dt.append(f1_score(y_test, pred_test))
    train_acc_dt.append(accuracy_score(y_train, pred_train))
    test_acc_dt.append(accuracy_score(y_test, pred_test))

best_depth = depths[np.argmax(f1_scores_dt)]
print(f"🏆 Best max_depth = {best_depth}")
print(f"   F1 = {max(f1_scores_dt):.4f}")

# Plot + overfitting demonstration
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(depths, f1_scores_dt, 'o-', color='green', linewidth=2)
axes[0].axvline(best_depth, color='red', linestyle=':', label=f'Best depth = {best_depth}')
axes[0].set_xlabel('max_depth'); axes[0].set_ylabel('F1 Score')
axes[0].set_title('Decision Tree: F1 vs depth', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(depths, train_acc_dt, 'o-', label='Train accuracy', color='steelblue')
axes[1].plot(depths, test_acc_dt, 's-', label='Test accuracy', color='orange')
axes[1].fill_between(depths, train_acc_dt, test_acc_dt, alpha=0.2, color='red')
axes[1].set_xlabel('max_depth'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Overfitting visualisation (gap = overfit)', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


> 🔬 **Observation:** notice how **train accuracy keeps increasing**, while **test accuracy stops improving** after the best depth. That is clear **overfitting**. The red area is the gap: memorisation without generalisation.


## Part 8 · Final comparison and IB-style conclusion


In [ ]:
# === Final models with optimal hyperparameters ===
best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn.fit(X_train_scaled, y_train)
y_pred_best_knn = best_knn.predict(X_test_scaled)

best_dt = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
best_dt.fit(X_train, y_train)
y_pred_best_dt = best_dt.predict(X_test)

# Summary table
final = pd.DataFrame({
    f'KNN (k={best_k})': [
        accuracy_score(y_test, y_pred_best_knn),
        precision_score(y_test, y_pred_best_knn),
        recall_score(y_test, y_pred_best_knn),
        f1_score(y_test, y_pred_best_knn),
    ],
    f'Decision Tree (depth={best_depth})': [
        accuracy_score(y_test, y_pred_best_dt),
        precision_score(y_test, y_pred_best_dt),
        recall_score(y_test, y_pred_best_dt),
        f1_score(y_test, y_pred_best_dt),
    ],
}, index=['Accuracy', 'Precision', 'Recall', 'F1']).round(4)

print("=" * 50)
print("🏆 FINAL RESULTS")
print("=" * 50)
print(final)

best_model = final.idxmax(axis=1).iloc[3]  # best by F1
print(f"\n🥇 Winner by F1: {best_model}")
print(f"   F1 = {final[best_model].iloc[3]:.4f}")


### 🎯 TRY IT #3 — Record your result

> ⚔️ Your Best F1 Score: **____________**
>
> 🏆 Write it on the board. Who is highest?


### 📝 IB-style conclusion (for an IA report)

> This is a **report template** you will use in the homework. Today, fill it in briefly.

```
1. PREPROCESSING (A4.2):
   - The filter method (Pearson's r) was applied for feature selection.
   - The feature `random_noise` was removed (|r| < 0.05) because it had no predictive value.
   - StandardScaler was applied only for KNN because Decision Trees do not require scaling.

2. MODELS TRAINED:
   - KNN with optimal K = ___ → F1 = ___
   - Decision Tree with optimal depth = ___ → F1 = ___

3. JUSTIFICATION:
   - Decision Tree provides stronger interpretability, which matters for a bank because
     regulators require loan decisions to be explainable.
   - KNN achieved [higher/lower] F1, making it [more/less] suitable.

4. RECOMMENDATION:
   For a production system, I recommend ___ because:
   (a) ___
   (b) ___
   (c) ___
```


## ✅ What we did in the workshop

- [x] **Feature selection** using a filter method (A4.2.2) ✓
- [x] **Train/Test split** with stratification and normalisation
- [x] Trained **KNN** (lazy learner) — A4.3.2 ✓
- [x] Trained **Decision Tree** (eager learner) — A4.3.2 ✓
- [x] Visualised the tree itself (interpretability!)
- [x] Calculated **all 4 metrics** + confusion matrix — A4.3.3 ✓
- [x] Performed **hyperparameter tuning** for K and max_depth — A4.3.3 ✓
- [x] Saw **overfitting** in train vs test plots
- [x] Compared models using **F1 score** (preview of A4.3.10)

---

### 🏠 Homework (see `Week2_HW2_Practice.ipynb`)

> Full **KNN vs Decision Tree** implementation on a **different** dataset + **IB-style Model Selection report**.

---

### 💎 Final workshop secrets

1. **KNN without normalisation = disaster.** Always use `StandardScaler` or `MinMaxScaler`.
2. **Decision Tree without `max_depth` = overfitting.** Always limit depth.
3. **`fit_transform` on train, `transform` on test**; otherwise data leakage = -1 IA mark.
4. **`stratify=y`** in `train_test_split` is critical for imbalanced classes.
5. **F1 score** > Accuracy for imbalanced classes; write this clearly in reports.
6. **`feature_importances_`** from Decision Tree is free feature selection. Link it to A4.2.2 embedded methods.
